In [1]:
import pandas as pd

TGIF_TSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\TGIF\tgif-v1.0.tsv"   # <-- update if needed
tgif = pd.read_csv(TGIF_TSV, sep="\t", header=None, names=["url", "ref_caption"])

print("Rows:", len(tgif))
print(tgif.head())

Rows: 125782
                                                 url  \
0  https://38.media.tumblr.com/9f6c25cc350f12aa74...   
1  https://38.media.tumblr.com/9ead028ef62004ef6a...   
2  https://38.media.tumblr.com/9f43dc410be85b1159...   
3  https://38.media.tumblr.com/9f659499c8754e40cf...   
4  https://38.media.tumblr.com/9ed1c99afa7d714118...   

                                         ref_caption  
0  a man is glaring, and someone with sunglasses ...  
1           a cat tries to catch a mouse on a tablet  
2                   a man dressed in red is dancing.  
3     an animal comes close to another in the jungle  
4  a man in a hat adjusts his tie and makes a wei...  


In [2]:
import os, hashlib
import requests

CACHE_DIR = "tgif_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def url_to_cache_path(url: str) -> str:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.gif")

def get_gif_bytes(url: str, use_cache=True, timeout=20) -> bytes:
    cache_path = url_to_cache_path(url)
    if use_cache and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            return f.read()

    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    data = r.content

    if use_cache:
        with open(cache_path, "wb") as f:
            f.write(data)
    return data

In [3]:
import sys

# If main_2.py is in a different folder, add it:

import main_2

# sanity: confirm these exist
needed = ["extract_middle_frame", "detect_content_type", "detect_action",
          "detect_emotion", "detect_objects", "generate_caption"]
for n in needed:
    print(n, "OK" if hasattr(main_2, n) else "MISSING")

 Using device: cpu
 Loading emotion detection model...
 Emotion model loaded!
 Loading object detection model...
 Object detector loaded!
 Loading action detection model...


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

 Action detector loaded!
extract_middle_frame OK
detect_content_type OK
detect_action OK
detect_emotion OK
detect_objects OK
generate_caption OK


In [4]:
import inspect, importlib, os

print("main_2 imported from:", main_2.__file__)
print("generate_caption source snippet:")
print("\n".join(inspect.getsource(main_2.generate_caption).splitlines()[:25]))

main_2 imported from: d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\TGIF\main_2.py
generate_caption source snippet:
def generate_caption(
    emotion: str,
    objects: Optional[List[str]] = None,
    action: Optional[str] = None,
    content_type: str = "real_world"
) -> str:
    """
    Emotion-aware TGIF-aligned captioning (emotion ALWAYS included):
    - Always anchors to a PERSON subject for real_world (TGIF-like)
    - Cartoon uses "an animated <adj> person ..."
    - Improved verb inflection (tripling, tying, dancing)
    - Inserts articles in action phrases (punching a bag, grooming a horse)
    - Smarter object prepositions: vehicles -> "near", others -> "with"
    - a/an article selection for object nouns
    - Avoids duplication like "petting cat with a cat"
    """

    objects = objects or []

    # -------------------------
    # 1) Clean / dedupe objects
    # -------------------------
    norm = []
    seen = set()
    for o in objects:


In [5]:
import importlib
import main_2
importlib.reload(main_2)

 Using device: cpu
 Loading emotion detection model...
 Emotion model loaded!
 Loading object detection model...
 Object detector loaded!
 Loading action detection model...


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

 Action detector loaded!


<module 'main_2' from 'd:\\IIT\\Year 4\\FYP\\Datasets\\GIFGIF_lucas\\TGIF\\main_2.py'>

In [6]:
import io
import numpy as np
from PIL import Image, ImageSequence

def extract_representative_frame_local(gif_bytes: bytes) -> Image.Image | None:
    """
    Simple and reliable: take the middle frame of the GIF.
    """
    try:
        gif = Image.open(io.BytesIO(gif_bytes))
        frames = [fr.convert("RGB") for fr in ImageSequence.Iterator(gif)]
        if len(frames) == 0:
            return None
        return frames[len(frames)//2]
    except Exception as e:
        print(" representative frame error:", e)
        return None
    
def normalize_emotion(emotion_out):
    """
    Convert emotion output into a string label.
    Handles:
      - "happiness"
      - ("happiness", 0.83)
      - {"label":"happiness","score":0.83}
      - None
    """
    if emotion_out is None:
        return "neutral"

    # tuple/list like (label, score)
    if isinstance(emotion_out, (tuple, list)) and len(emotion_out) > 0:
        return str(emotion_out[0])

    # dict like {"label":..., "score":...}
    if isinstance(emotion_out, dict):
        if "label" in emotion_out:
            return str(emotion_out["label"])

    # already a string or something else
    return str(emotion_out)

def detect_objects_wrapper(frame: Image.Image):
    """
    Calls whatever object detection function exists in main_2.
    Returns list[str] (object names) or [].
    """
    # Try common names
    for fn_name in ["detect_objects_in_frame", "detect_objects", "detect_objects_yolo", "get_objects"]:
        if hasattr(main_2, fn_name):
            try:
                out = getattr(main_2, fn_name)(frame)
                if out is None:
                    return []
                return out
            except Exception as e:
                print(f" object fn {fn_name} failed:", e)
                return []
    # If no function found, return empty
    print("️ No object detection function found in main_2.py")
    return []

In [7]:
import io
import numpy as np
from PIL import Image, ImageSequence
from collections import Counter

def extract_k_frames_evenly(gif_bytes: bytes, k: int = 5):
    """
    Extract k evenly spaced RGB frames from a GIF (bytes).
    Returns: list[PIL.Image.Image]
    """
    gif = Image.open(io.BytesIO(gif_bytes))
    frames = [fr.convert("RGB") for fr in ImageSequence.Iterator(gif)]
    if not frames:
        return []
    if k <= 1:
        return [frames[len(frames)//2]]
    idxs = np.linspace(0, len(frames) - 1, k, dtype=int)
    return [frames[i] for i in idxs]

from collections import Counter

def detect_objects_multiframe_vote(
    gif_bytes: bytes,
    k: int = 5,
    top_n: int = 2,
    min_votes: int = 2,
    drop_labels: set | None = None
):
    if drop_labels is None:
        drop_labels = {"person", "man", "woman", "people", "human"}

    frames = extract_k_frames_evenly(gif_bytes, k=k)
    if not frames:
        return []

    all_labels = []
    for fr in frames:
        objs = detect_objects_wrapper(fr) or []
        normed = []
        for o in objs:
            oo = str(o).strip().lower()
            if oo and oo not in drop_labels:
                normed.append(oo)
        # keep top2 per frame
        all_labels.extend(normed[:2])

    # If nothing detected across frames, fallback to middle frame
    if not all_labels:
        mid = frames[len(frames)//2]
        fb = [o.lower() for o in (detect_objects_wrapper(mid) or []) if o]
        fb = [o for o in fb if o not in drop_labels]
        return fb[:top_n]

    counts = Counter(all_labels)
    winners = [(lbl, c) for lbl, c in counts.most_common() if c >= min_votes]

    # If voting too strict, fallback to MOST COMMON overall (even if only 1 vote)
    if not winners:
        winners = counts.most_common(top_n)

    return [lbl for lbl, _ in winners[:top_n]]

def clean_objects_for_caption(objs):
    if not objs:
        return []
    # remove very weird common nuisances for TGIF
    block = {"traffic light", "tie", "baseball glove", "tennis racket", "airplane"}
    out = []
    for o in objs:
        oo = o.strip().lower()
        if oo and oo not in block:
            out.append(oo)
    return out[:2]

def insert_article_in_action(action, objects):
    if not action:
        return action

    a = action.lower().strip()
    parts = a.split()
    if len(parts) == 2:
        verb, noun = parts

        # if noun is in detected objects, add article
        obj_set = set([o.lower() for o in (objects or [])])
        if noun in obj_set:
            return f"{verb} a {noun}"

        # animals fallback
        animals = {"dog","cat","horse","bird","bear","elephant","giraffe","zebra","lion","tiger","cow","sheep","monkey","rabbit"}
        if noun in animals:
            return f"{verb} a {noun}"

        # common action objects in TGIF-like captions
        common_nouns = {"bag","ball","pipe","phone","camera","guitar","horse","dog","cat","door","car","bike","bicycle","motorcycle","airplane"}
        if noun in common_nouns:
            return f"{verb} a {noun}"

    return action

def choose_object_preposition(obj: str) -> str:
    vehicles = {"airplane","motorcycle","car","bus","truck","train","boat","ship","bicycle","bike"}
    o = (obj or "").lower().strip()
    return "near" if o in vehicles else "with"

In [8]:
def predict_caption_from_gifbytes(gif_bytes: bytes):
    # 1) representative frame
    frame = extract_representative_frame_local(gif_bytes)
    if frame is None:
        return "", [], None, None   # (caption, objects, action, content_type)

    # 2) content type
    content_type, ct_conf = main_2.detect_content_type(frame)

    # 3) objects (MULTI-FRAME VOTING)
    objects = detect_objects_multiframe_vote(gif_bytes, k=3, top_n=1)
    objects = clean_objects_for_caption(objects)

    # 4) action (VideoMAE over multiple frames)
    action = None
    if content_type == "real_world":
        action = main_2.detect_action(gif_bytes)

    # grammar helper
    action = insert_article_in_action(action, objects)

    # 5) emotion
    try:
        emo_out = main_2.detect_emotion(frame)
    except Exception as e:
        print(" detect_emotion failed:", e)
        emo_out = None

    emotion = normalize_emotion(emo_out)

    # 6) caption
    try:
        print(
            "DEBUG objects voted:", objects,
            "| action:", action,
            "| content:", content_type,
            "| emotion:", emotion
        )

        cap = main_2.generate_caption(
            emotion=emotion,
            objects=objects,
            action=action,
            content_type=content_type
        )
        return cap or "", objects, action, content_type

    except Exception as e:
        print(" generate_caption failed:", e)
        return "", objects, action, content_type

In [9]:
row = tgif.iloc[0]
gif_bytes = get_gif_bytes(row["url"])
pred = predict_caption_from_gifbytes(gif_bytes)
print("PRED:", pred)
print("REF :", row["ref_caption"])

 Content Detection - Color: 0.001, Sat: 0.074, Face: False
   → Cartoon (extremely low colors)
DEBUG objects voted: [] | action: None | content: cartoon | emotion: negative_subdued
 Generated caption: 'an animated dejected person is looking' (emotion: negative_subdued, objects: [], action: None, type: cartoon)
PRED: ('an animated dejected person is looking', [], None, 'cartoon')
REF : a man is glaring, and someone with sunglasses appears.


In [12]:
import re
import numpy as np
from tqdm import tqdm

# BLEU (NLTK)
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# METEOR (NLTK)
from nltk.translate.meteor_score import meteor_score

# Make sure required NLTK resources exist
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

smooth = SmoothingFunction().method1

def tokenize(s: str):
    s = (s or "").lower().strip()
    # simple tokenization: words + numbers
    return re.findall(r"[a-z0-9']+", s)

# ---------------------------
# ROUGE (pure python)
# ROUGE-1 / ROUGE-2: n-gram overlap F1
# ROUGE-L: LCS-based F1
# ---------------------------

def _ngrams(tokens, n):
    if n <= 0:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def rouge_n_f1(pred_toks, ref_toks, n=1):
    pred_ngr = _ngrams(pred_toks, n)
    ref_ngr  = _ngrams(ref_toks, n)
    if len(pred_ngr) == 0 or len(ref_ngr) == 0:
        return 0.0

    # count overlap with multiplicity
    from collections import Counter
    pc = Counter(pred_ngr)
    rc = Counter(ref_ngr)
    overlap = sum((pc & rc).values())

    prec = overlap / max(1, len(pred_ngr))
    rec  = overlap / max(1, len(ref_ngr))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def _lcs_len(a, b):
    # classic DP LCS length (O(len(a)*len(b))) – OK for short captions
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    dp = [0] * (n + 1)
    for i in range(1, m + 1):
        prev = 0
        for j in range(1, n + 1):
            tmp = dp[j]
            if a[i-1] == b[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = tmp
    return dp[n]

def rouge_l_f1(pred_toks, ref_toks):
    if len(pred_toks) == 0 or len(ref_toks) == 0:
        return 0.0
    lcs = _lcs_len(pred_toks, ref_toks)
    prec = lcs / max(1, len(pred_toks))
    rec  = lcs / max(1, len(ref_toks))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def compute_metrics(pred: str, ref: str):
    pred_toks = tokenize(pred)
    ref_toks  = tokenize(ref)

    # BLEU-1..4
    b1 = sentence_bleu([ref_toks], pred_toks, weights=(1,0,0,0), smoothing_function=smooth)
    b2 = sentence_bleu([ref_toks], pred_toks, weights=(0.5,0.5,0,0), smoothing_function=smooth)
    b3 = sentence_bleu([ref_toks], pred_toks, weights=(1/3,1/3,1/3,0), smoothing_function=smooth)
    b4 = sentence_bleu([ref_toks], pred_toks, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth)

    # METEOR 
    met = meteor_score([ref_toks], pred_toks)

    # ROUGE
    r1 = rouge_n_f1(pred_toks, ref_toks, n=1)
    r2 = rouge_n_f1(pred_toks, ref_toks, n=2)
    rl = rouge_l_f1(pred_toks, ref_toks)

    return b1, b2, b3, b4, met, r1, r2, rl


In [13]:
empty_obj = 0
empty_act = 0

N = 50
rows = tgif.sample(N, random_state=0).reset_index(drop=True)

bleu1s, bleu2s, bleu3s, bleu4s = [], [], [], []
mets = []
r1s, r2s, rls = [], [], []

examples = []

for i, row in tqdm(rows.iterrows(), total=len(rows)):
    url = row["url"]
    ref = row["ref_caption"]

    gif_bytes = get_gif_bytes(url)
    if gif_bytes is None:
        continue

    pred, objects, action, content_type = predict_caption_from_gifbytes(gif_bytes)

    if not objects:
        empty_obj += 1
    if not action:
        empty_act += 1

    b1, b2, b3, b4, met, r1, r2, rl = compute_metrics(pred, ref)
    bleu1s.append(b1); bleu2s.append(b2); bleu3s.append(b3); bleu4s.append(b4)
    mets.append(met)
    r1s.append(r1); r2s.append(r2); rls.append(rl)

    if i < 8:
        examples.append((pred, ref))

print("Samples scored:", len(bleu1s))
print("Empty objects:", empty_obj, "/", len(bleu1s))
print("Empty actions:", empty_act, "/", len(bleu1s))

print("\n--- N-gram metrics (expected to be low due to emotion injection + single refs) ---")
print("BLEU-1 :", float(np.mean(bleu1s)))
print("BLEU-2 :", float(np.mean(bleu2s)))
print("BLEU-3 :", float(np.mean(bleu3s)))
print("BLEU-4 :", float(np.mean(bleu4s)))
print("METEOR :", float(np.mean(mets)))

print("\n--- ROUGE (F1) ---")
print("ROUGE-1:", float(np.mean(r1s)))
print("ROUGE-2:", float(np.mean(r2s)))
print("ROUGE-L:", float(np.mean(rls)))

print("\nExamples:")
for pred, ref in examples:
    print("\nPRED:", pred)
    print("REF :", ref)


  0%|          | 0/50 [00:00<?, ?it/s]

 Content Detection - Color: 0.003, Sat: 0.721, Face: False
   → Cartoon (very low colors + saturated)
DEBUG objects voted: [] | action: None | content: cartoon | emotion: positive_energetic
 Generated caption: 'an animated happy person is playing' (emotion: positive_energetic, objects: [], action: None, type: cartoon)


  2%|▏         | 1/50 [00:03<02:46,  3.39s/it]

 Content Detection - Color: 0.002, Sat: 0.036, Face: True
   → Real-world (face detected)


  4%|▍         | 2/50 [00:07<03:07,  3.90s/it]

...

DEBUG objects voted: ['cat'] | action: None | content: cartoon | emotion: positive_energetic
 Generated caption: 'an animated happy person is playing with a cat' (emotion: positive_energetic, objects: ['cat'], action: None, type: cartoon)
 Content Detection - Color: 0.001, Sat: 0.000, Face: True
   → Real-world (face detected)


 98%|█████████▊| 49/50 [01:24<00:01,  1.62s/it]

DEBUG objects voted: [] | action: None | content: real_world | emotion: positive_calm
 Generated caption: 'a serene person is relaxing' (emotion: positive_calm, objects: [], action: None, type: real_world)
 Content Detection - Color: 0.000, Sat: 0.000, Face: True
   → Real-world (face detected)


100%|██████████| 50/50 [01:27<00:00,  1.74s/it]

DEBUG objects voted: ['cat'] | action: petting a cat | content: real_world | emotion: positive_energetic
 Generated caption: 'an energetic person is petting a cat' (emotion: positive_energetic, objects: ['cat'], action: petting a cat, type: real_world)
Samples scored: 50
Empty objects: 33 / 50
Empty actions: 38 / 50

--- N-gram metrics (expected to be low due to emotion injection + single refs) ---
BLEU-1 : 0.1280143429395586
BLEU-2 : 0.047900000335301514
BLEU-3 : 0.026831657111816277
BLEU-4 : 0.02191599600114408
METEOR : 0.09038306857774289

--- ROUGE (F1) ---
ROUGE-1: 0.17817600957693838
ROUGE-2: 0.01800928070897111
ROUGE-L: 0.16392415444892222

Examples:

PRED: an animated happy person is playing
REF : a female athlete is performing the long jump.

PRED: a lonely person is waiting
REF : a man is dancing and moving in front of the cameras

PRED: an animated disgusted person is crying
REF : a woman is lighting and smoking from a pipe

PRED: a lonely person is applying cream
REF : a wo